# 07 — Model Comparison

Compare all 6 generative models on synthetic chest X-ray generation.

**Models:**
1. Stable Diffusion (img2img)
2. DCGAN
3. WGAN-GP
4. VQ-VAE
5. DDPM
6. Flow Matching

**Metrics:**
- Domain-adapted FID (DenseNet121 features)
- Proxy TSTR accuracy (torchxrayvision)

## Setup

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
OUTPUTS_DIR = Path("outputs")

MODEL_DIRS = {
    'Stable Diffusion': OUTPUTS_DIR / '01_stable_diffusion',
    'DCGAN': OUTPUTS_DIR / '02_dcgan',
    'WGAN-GP': OUTPUTS_DIR / '03_wgan_gp',
    'VQ-VAE': OUTPUTS_DIR / '04_vqvae',
    'DDPM': OUTPUTS_DIR / '05_ddpm',
    'Flow Matching': OUTPUTS_DIR / '06_flow_matching'
}

## Load Metrics

In [ ]:
def load_metrics(model_dir):
    """Load metrics.json from model output directory."""
    metrics_path = model_dir / 'metrics.json'
    if metrics_path.exists():
        with open(metrics_path) as f:
            return json.load(f)
    return None

In [ ]:
all_metrics = {}

for name, model_dir in MODEL_DIRS.items():
    metrics = load_metrics(model_dir)
    if metrics:
        all_metrics[name] = metrics
        print(f"✓ {name}: FID={metrics.get('fid_domain_adapted', 'N/A'):.2f}, TSTR={metrics.get('tstr_accuracy', 'N/A'):.1f}%")
    else:
        print(f"✗ {name}: Not trained yet")

print(f"\nModels available: {len(all_metrics)}/6")

## Comparison Table

In [ ]:
if all_metrics:
    comparison_data = []
    
    for name, metrics in all_metrics.items():
        comparison_data.append({
            'Model': name,
            'FID (↓)': metrics.get('fid_domain_adapted'),
            'TSTR (↑)': metrics.get('tstr_accuracy'),
            'Mean Pneumonia Score': metrics.get('mean_pneumonia_score'),
            'Epochs': metrics.get('epochs', 'N/A'),
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('FID (↓)')
    
    print("\n" + "="*70)
    print("MODEL COMPARISON")
    print("="*70)
    print(comparison_df.to_string(index=False))
    print("="*70)
    print("FID: Lower is better (domain-adapted, DenseNet121 features)")
    print("TSTR: Higher is better (proxy accuracy using torchxrayvision)")
else:
    print("No models trained yet.")

## FID Comparison Chart

In [ ]:
if all_metrics:
    models = list(all_metrics.keys())
    fid_scores = [all_metrics[m].get('fid_domain_adapted', 0) for m in models]
    
    # sort by FID
    sorted_pairs = sorted(zip(models, fid_scores), key=lambda x: x[1])
    models_sorted = [p[0] for p in sorted_pairs]
    fid_sorted = [p[1] for p in sorted_pairs]
    
    plt.figure(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(models_sorted)))
    bars = plt.barh(models_sorted, fid_sorted, color=colors)
    
    plt.xlabel('Domain-adapted FID (lower is better)')
    plt.title('FID Comparison Across Models')
    
    # add values on bars
    for bar, val in zip(bars, fid_sorted):
        if val:
            plt.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
                    f'{val:.1f}', va='center')
    
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / 'fid_comparison.png', dpi=150)
    plt.show()

## TSTR Comparison Chart

In [ ]:
if all_metrics:
    tstr_scores = [all_metrics[m].get('tstr_accuracy', 0) for m in models]
    
    # sort by TSTR (descending - higher is better)
    sorted_pairs = sorted(zip(models, tstr_scores), key=lambda x: x[1], reverse=True)
    models_sorted = [p[0] for p in sorted_pairs]
    tstr_sorted = [p[1] for p in sorted_pairs]
    
    plt.figure(figsize=(10, 6))
    colors = plt.cm.plasma(np.linspace(0.2, 0.8, len(models_sorted)))
    bars = plt.barh(models_sorted, tstr_sorted, color=colors)
    
    plt.xlabel('Proxy TSTR Accuracy % (higher is better)')
    plt.title('TSTR Comparison Across Models')
    plt.xlim(0, 100)
    
    # add values on bars
    for bar, val in zip(bars, tstr_sorted):
        if val:
            plt.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
                    f'{val:.1f}%', va='center')
    
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / 'tstr_comparison.png', dpi=150)
    plt.show()

## Sample Images from Each Model

In [ ]:
n_samples = 5
n_models = len(all_metrics)

if n_models > 0:
    fig, axes = plt.subplots(n_models, n_samples, figsize=(n_samples * 2, n_models * 2))
    
    if n_models == 1:
        axes = axes.reshape(1, -1)
    
    for i, (name, metrics) in enumerate(all_metrics.items()):
        images_dir = MODEL_DIRS[name] / 'images'
        if images_dir.exists():
            image_paths = sorted(images_dir.glob('*.png'))[:n_samples]
            for j, img_path in enumerate(image_paths):
                img = Image.open(img_path)
                axes[i, j].imshow(img, cmap='gray')
                axes[i, j].axis('off')
                if j == 0:
                    axes[i, j].set_ylabel(name, fontsize=10, rotation=0, ha='right', va='center')
        
        # label row
        axes[i, 0].set_title(f"{name}\nFID: {metrics.get('fid_domain_adapted', 'N/A'):.1f}", 
                            fontsize=9, loc='left')
    
    plt.suptitle('Generated Samples from Each Model', fontsize=14)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / 'samples_comparison.png', dpi=150)
    plt.show()

## Summary

In [ ]:
if all_metrics:
    best_fid = min(all_metrics.items(), key=lambda x: x[1].get('fid_domain_adapted', float('inf')))
    best_tstr = max(all_metrics.items(), key=lambda x: x[1].get('tstr_accuracy', 0))
    
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    print(f"\nBest FID: {best_fid[0]} ({best_fid[1].get('fid_domain_adapted', 'N/A'):.2f})")
    print(f"Best TSTR: {best_tstr[0]} ({best_tstr[1].get('tstr_accuracy', 'N/A'):.1f}%)")
    
    print("\n" + "-"*60)
    print("Notes:")
    print("-"*60)
    print("- FID uses domain-adapted DenseNet121 features (not Inception V3)")
    print("- TSTR is a proxy metric using torchxrayvision classifier")
    print("- Stable Diffusion uses img2img (different input distribution)")
    print("- VQ-VAE uses random codebook sampling (no PixelCNN prior)")
    print("="*60)
else:
    print("Run the model notebooks first to generate metrics.")

## Export Combined Metrics

In [ ]:
if all_metrics:
    combined_metrics = {
        'models': all_metrics,
        'best_fid': {
            'model': best_fid[0],
            'score': best_fid[1].get('fid_domain_adapted')
        },
        'best_tstr': {
            'model': best_tstr[0],
            'score': best_tstr[1].get('tstr_accuracy')
        }
    }
    
    with open(OUTPUTS_DIR / 'combined_metrics.json', 'w') as f:
        json.dump(combined_metrics, f, indent=2)
    
    print(f"Saved combined metrics to {OUTPUTS_DIR / 'combined_metrics.json'}")
    
    # also save comparison table as CSV
    comparison_df.to_csv(OUTPUTS_DIR / 'comparison_table.csv', index=False)
    print(f"Saved comparison table to {OUTPUTS_DIR / 'comparison_table.csv'}")